# EDA - Exploratory Data Analysis do dataset CIC-YNU-IoTMal 2026
Mesmos passos do tutorial da WineQT, adaptados a este dataset (737 mil linhas).

Fonte: [IoT Malware Network Traffic Dataset (CIC-YNU-2026)](https://www.kaggle.com/datasets/raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026), carregado diretamente do Kaggle (sem ficheiro local).

## Passo 0: Descarregar o dataset do Kaggle
`kagglehub` já vem instalado no Colab. Se a célula seguinte pedir autenticação
(mensagem a pedir login), corre a célula depois dela com `kagglehub.login()`
e segue o link que aparece — depois volta a correr esta célula.

In [ ]:
%pip install -q kagglehub

import kagglehub

DATASET = "raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026"
dataset_path = kagglehub.dataset_download(DATASET)
print("Dataset descarregado em:", dataset_path)

**Só corre a célula abaixo se a anterior pedir autenticação (login no Kaggle):**

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
# Encontrar o(s) ficheiro(s) CSV dentro da pasta descarregada
import glob, os

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("CSVs encontrados:")
for f in csv_files:
    print(" -", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

CSV_PATH = csv_files[0]
print("\nA usar:", CSV_PATH)

## Passo 1: Importar bibliotecas

In [ ]:
import warnings as wr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

wr.filterwarnings("ignore")
%matplotlib inline

TARGET = "MalwareFamily"
SEED = 42

## Passo 2: Ler o dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

## Passo 3: Analisar os dados

In [ ]:
# 3.1 shape
df.shape

In [ ]:
# 3.2 info
df.info()

In [ ]:
# 3.3 describe
pd.set_option("display.width", 250)
df.describe().T

In [ ]:
# 3.4 colunas
df.columns.tolist()

## Passo 4: Valores em falta

In [ ]:
print(df.isnull().sum()[lambda s: s > 0])

# Rate tem valores infinitos -> tratar como NaN
print("Rate com inf:", np.isinf(df["Rate"]).sum())
df["Rate"] = df["Rate"].replace([np.inf, -np.inf], np.nan)

## Passo 5: Duplicados

In [ ]:
n_dup = df.duplicated().sum()
print(f"Linhas duplicadas: {n_dup} ({n_dup / len(df) * 100:.2f}%)")

## Extras: variável alvo, colunas constantes/redundantes e amostra

In [ ]:
print(df[TARGET].value_counts())
print("\nHashes únicos:", df["Hash"].nunique())

num_all = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
constantes = [c for c in num_all if df[c].nunique() == 1]
redundantes = [c for c in ["Tot size", "LLC"] if c in num_all]  # Tot size == AVG, LLC == IPv
print("Constantes:", constantes)
print("Redundantes:", redundantes)
numerical_columns = [c for c in num_all if c not in constantes + redundantes]

order = df[TARGET].value_counts().index.tolist()


def amostra_estratificada(data, por_classe=3000):
    partes = [g.sample(min(len(g), por_classe), random_state=SEED) for _, g in data.groupby(TARGET)]
    return pd.concat(partes).reset_index(drop=True)


amostra = amostra_estratificada(df)

## Passo 6: Análise univariada

In [ ]:
# 6.1 Bar plot da variável alvo
counts = df[TARGET].value_counts()
plt.figure(figsize=(9, 6))
bars = plt.bar(counts.index, counts.values, color="deeppink")
plt.yscale("log")
plt.title("Count Plot de MalwareFamily (escala log)")
plt.xlabel("MalwareFamily")
plt.ylabel("Contagem")
for b, v in zip(bars, counts.values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.show()

In [ ]:
# 6.2 Histogramas + KDE com skewness (skew nos dados todos, KDE numa amostra)
sns.set_style("darkgrid")
kde_sample = df.sample(50_000, random_state=SEED)
ncols = 3
nrows = int(np.ceil(len(numerical_columns) / ncols))
plt.figure(figsize=(15, nrows * 2.6))
for idx, feature in enumerate(numerical_columns, 1):
    plt.subplot(nrows, ncols, idx)
    sns.histplot(kde_sample[feature].dropna(), kde=True, bins=40)
    plt.title(f"{feature} | Skewness: {round(df[feature].skew(), 2)}", fontsize=9)
    plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 Outliers: swarm plot (amostra) + box plot (dados completos)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
sns.swarmplot(y="Tot sum", data=df.sample(1500, random_state=SEED), color="skyblue", size=2.5, ax=axes[0])
axes[0].set_title("Swarm Plot de Tot sum (amostra de 1500)")
sns.boxplot(y="Tot sum", data=df, color="skyblue", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Box Plot de Tot sum (todos os dados, escala log)")
plt.tight_layout()
plt.show()

## Passo 7: Análise bivariada

In [ ]:
# 7.1 Pair plot (amostra, features em log1p)
pp = amostra[[TARGET]].copy()
for c in ["Rate", "Tot sum", "IAT", "Header_Length", "Time_To_Live", "Std"]:
    pp[f"log1p({c})"] = np.log1p(amostra[c])
sns.set_palette("tab10")
g = sns.pairplot(pp, hue=TARGET, hue_order=order, corner=True,
                 plot_kws={"s": 8, "alpha": 0.5}, diag_kind="hist")
g.fig.suptitle("Pair Plot (amostra estratificada, features em log1p)", y=1.01)
plt.show()

In [ ]:
# 7.2 / 7.3 Violin e Box plot vs. família
sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for j, f in enumerate(["Rate", "AVG"]):
    tmp = amostra[[TARGET, f]].copy()
    tmp[f] = np.log10(tmp[f].where(tmp[f] > 0))
    lab = f"log10({f})"
    sns.violinplot(x=TARGET, y=f, data=tmp, order=order, hue=TARGET,
                   palette="Set2", legend=False, ax=axes[0, j], cut=0)
    axes[0, j].set_title(f"Violin Plot: {lab} por MalwareFamily")
    axes[0, j].set_ylabel(lab)
    sns.boxplot(x=TARGET, y=f, data=tmp, order=order, hue=TARGET,
                palette="Set2", legend=False, ax=axes[1, j])
    axes[1, j].set_title(f"Box Plot: {lab} por MalwareFamily")
    axes[1, j].set_ylabel(lab)
plt.tight_layout()
plt.show()

## Passo 8: Análise multivariada

In [ ]:
corr = df[numerical_columns].corr()
plt.figure(figsize=(22, 17))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, annot_kws={"size": 5})
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Pares muito correlacionados (|r| > 0.95)
tri = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
tri[tri.abs() > 0.95].sort_values(ascending=False).round(3)

# Passo 9: Preparação dos Dados (Data Preparation)

Agora que a EDA está feita, preparamos os dados para o `scikit-learn`.
Cada célula corresponde a um dos pontos do módulo "Data Preparation":
limpeza, valores em falta, outliers, divisão treino/teste, seleção de
features, scaling e desequilíbrio de classes.

## 9.1 Limpeza de dados
Remove duplicados, colunas constantes e colunas repetidas identificadas na EDA.

In [ ]:
df_prep = df.copy()

# Rate tinha valores infinitos -> já tratados como NaN no Passo 4, repetimos aqui por segurança
df_prep["Rate"] = df_prep["Rate"].replace([np.inf, -np.inf], np.nan)

# duplicados
antes = len(df_prep)
df_prep = df_prep.drop_duplicates()
print(f"Duplicados removidos: {antes - len(df_prep)}")

# colunas constantes (incluindo 'Arch', que é sempre 'arm' mas é texto, por isso escapa ao .describe() numérico)
# e colunas redundantes (Tot size == AVG, LLC == IPv)
colunas_constantes = ["SMTP", "SSH", "IRC", "Arch"]
colunas_redundantes = ["Tot size", "LLC"]
df_prep = df_prep.drop(columns=colunas_constantes + colunas_redundantes)

print("Shape após limpeza:", df_prep.shape)

## 9.2 Valores em falta
`Std`, `Variance` e `Rate` têm nulos nas mesmas 563 linhas (onde `Number == 1`, só há um pacote). É menos de 0,1% dos dados, por isso removemos essas linhas.

In [ ]:
print("Nulos antes:", df_prep.isnull().sum().sum())
df_prep = df_prep.dropna()
print("Shape após remover nulos:", df_prep.shape)

## 9.3 Outliers
Em tráfego de rede, valores extremos costumam ser ataques genuínos — por isso não os removemos. Aplicamos `log1p` às colunas mais assimétricas (vistas no Passo 6.2) para reduzir o seu peso sem as apagar.

In [ ]:
colunas_assimetricas = ["Rate", "Tot sum", "AVG", "Std", "Variance", "Max"]
for c in colunas_assimetricas:
    df_prep[c] = np.log1p(df_prep[c])

print("Skewness depois do log1p:")
print(df_prep[colunas_assimetricas].skew().round(2))

## 9.4 Divisão em treino e teste
Cada `Hash` tem várias linhas, todas da mesma família — se ficarem espalhadas
por treino e teste, o modelo "decora" o hash em vez de aprender o padrão.
Usamos `StratifiedGroupKFold`, que agrupa por `Hash` **e** tenta manter a
proporção de cada família semelhante nos dois conjuntos.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

TARGET = "MalwareFamily"
SEED = 42

X = df_prep.drop(columns=[TARGET, "Hash"])
y = df_prep[TARGET]
groups = df_prep["Hash"]

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Treino:", X_train.shape, " Teste:", X_test.shape)
print("Hashes em comum entre treino e teste (deve ser 0):",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

print("\nProporção de classes no treino:")
print(y_train.value_counts(normalize=True).round(3))
print("\nProporção de classes no teste:")
print(y_test.value_counts(normalize=True).round(3))

## 9.5 Seleção de features
Removemos um elemento de cada par com correlação acima de 0.95, identificado
no heatmap do Passo 8: os pares `_flag_number` / `_count` (mantemos a flag),
`UDP`/`DHCP` (mantemos UDP) e `ARP`/`IPv` (mantemos IPv, que já não é o `LLC`
duplicado).

In [ ]:
colunas_redundantes_corr = ["ack_count", "rst_count", "syn_count", "fin_count", "DHCP", "ARP"]

X_train = X_train.drop(columns=colunas_redundantes_corr)
X_test = X_test.drop(columns=colunas_redundantes_corr)

print("Features finais:", X_train.shape[1])
print(X_train.columns.tolist())

## 9.6 Normalização (scaling)
Usamos `RobustScaler`, mais resistente a outliers que o `StandardScaler`
(usa mediana e IQR em vez de média e desvio padrão). **Importante:** o
`scaler` ajusta-se (`fit`) só com o treino, e depois só se aplica
(`transform`) ao teste — nunca o contrário, para não haver fuga de
informação do teste para o treino.

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

## 9.7 Desequilíbrio de classes
Mirai domina o dataset (≈69%) e Agent/Tsunami quase não têm amostras.
`class_weight="balanced"` diz ao modelo para dar mais peso aos erros nas
classes raras. Isto é só uma verificação rápida de que o pipeline inteiro
funciona de ponta a ponta — a escolha e afinação do modelo final é a fase
seguinte do CRISP-ML (Modeling), não desta fase.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

clf = DecisionTreeClassifier(class_weight="balanced", max_depth=5, random_state=SEED)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print(classification_report(y_test, y_pred, zero_division=0))

**Nota sobre o resultado acima:** com `class_weight="balanced"`, a accuracy
geral pode parecer baixa — é esperado, porque o modelo está a sacrificar
acerto nas classes maioritárias (Mirai) para tentar aprender as minoritárias
(Agent, Tsunami). Repara antes nas métricas **por classe** (`precision`,
`recall`, `f1-score`) e no `macro avg`, que é o que importa comparar entre
diferentes modelos nesta fase seguinte do projeto.